# 🚀 Qiskit ハンドオフ — 同じ問題を本物の量子回路で

Tutorial / Challenge で触った 3 つのつまみ (**短さの好み γ / 混ぜる強さ β / 考え直す回数 reps**) を、本物の **Qiskit + AerSimulator** で動かすノートブックです。

## 進め方
1. Web アプリの「🐍 Qiskit でも試す」ボタンを押すと、パラメータがクリップボードにコピーされます
2. このノートブックの **最初のコードセル** に Cmd/Ctrl+V で貼り付ける
3. メニュー: **ランタイム → すべて実行**
4. 3〜5 分で全セル完走。最後に JS エンジンの結果と比較できます

**所要時間**: 約 5 分
**前提知識**: 不要 (実行するだけで OK)

## ① パラメータ貼り付けセル

Web アプリで `🐍 Qiskit でも試す` ボタンを押すと、↓ のような Python コードがクリップボードにコピーされます。次のセル全体を選択 → ペーストで上書きしてください。

(何もしない場合は下記のデフォルト値で動きます)

In [ ]:
# ============================================
# Web アプリからのパラメータ
# ============================================
GAMMA = 1.0
BETA = 0.4
REPS = 2
PROFILE = "midday"
TEAM_NAME = ""
LABELS = None
JS_TOP_ROUTES = []
# ============================================
print(f"γ={GAMMA}, β={BETA}, reps={REPS}, profile={PROFILE}")
if TEAM_NAME:
    print(f"Team: {TEAM_NAME}")

## ② ライブラリのインストール (初回のみ)

Colab には Qiskit が標準で入っていないので、最初にインストールします。1〜2 分かかります。

In [ ]:
!pip install -q qiskit==1.2.4 qiskit-aer==0.15.1 numpy matplotlib pandas
print("✅ install 完了")

## ③ 街のレイアウト (Web アプリと完全に同じ)

Web アプリの `src/engine/city-layout.ts` をそのまま Python に移植したセルです。

- 5×5 = **25 個の交差点**
- **6 つの配送先** + 倉庫 (中央)
- **40 本の道路エッジ**
- **4 段階の渋滞** (空き=×1.0, 少し=×1.25, 混み=×1.7, 渋滞=×2.4)

In [ ]:
import numpy as np
from itertools import permutations

GRID_VALUES = [-6, -3, 0, 3, 6]
DEPOT_ID = 'depot'

DELIVERY_SPECS = [
    {'gx': -6, 'gy':  6, 'label': '北西工場'},
    {'gx':  6, 'gy':  6, 'label': '北東病院'},
    {'gx':  6, 'gy':  0, 'label': '東モール'},
    {'gx': -6, 'gy':  0, 'label': '西駅'},
    {'gx': -6, 'gy': -6, 'label': '南西商店街'},
    {'gx':  6, 'gy': -6, 'label': '南東スタジアム'},
]

# チームがラベルを上書きしていた場合は反映
if LABELS and len(LABELS) == len(DELIVERY_SPECS):
    for i, lbl in enumerate(LABELS):
        DELIVERY_SPECS[i]['label'] = lbl

TRAFFIC_MULTIPLIER = {
    'clear':    1.0,
    'light':    1.25,
    'moderate': 1.7,
    'heavy':    2.4,
}

def node_id_for_grid(gx, gy):
    if gx == 0 and gy == 0:
        return DEPOT_ID
    return f'n_{gx}_{gy}'

def edge_id(a, b):
    return f'{a}~{b}' if a < b else f'{b}~{a}'

def fnv1a(key):
    h = 2166136261
    for ch in key:
        h = (h ^ ord(ch)) & 0xFFFFFFFF
        h = (h * 16777619) & 0xFFFFFFFF
    return h

def traffic_for_edge(edge_key, profile):
    h = fnv1a(edge_key)
    shift = {'morning_rush': 0, 'midday': 7, 'evening_rush': 13}[profile]
    slot = (h + shift) % 100
    if profile == 'morning_rush':
        return 'heavy' if slot < 18 else ('moderate' if slot < 50 else ('light' if slot < 78 else 'clear'))
    if profile == 'evening_rush':
        return 'heavy' if slot < 14 else ('moderate' if slot < 42 else ('light' if slot < 74 else 'clear'))
    return 'heavy' if slot < 6 else ('moderate' if slot < 24 else ('light' if slot < 62 else 'clear'))

# ノードリスト
nodes = []
delivery_node_ids = []
for gy in GRID_VALUES:
    for gx in GRID_VALUES:
        nid = node_id_for_grid(gx, gy)
        kind = 'depot' if (gx == 0 and gy == 0) else 'intersection'
        label = '倉庫' if kind == 'depot' else None
        for j, spec in enumerate(DELIVERY_SPECS):
            if spec['gx'] == gx and spec['gy'] == gy:
                kind = 'delivery'
                label = spec['label']
                delivery_node_ids.append(nid)
                break
        nodes.append({'id': nid, 'x': gx, 'y': gy, 'kind': kind, 'label': label})

# エッジ + 渋滞
by_coord = {(n['x'], n['y']): n for n in nodes}
edges = []
spacing = 3
for n in nodes:
    east = by_coord.get((n['x'] + spacing, n['y']))
    if east:
        eid = edge_id(n['id'], east['id'])
        edges.append({
            'id': eid, 'from': n['id'], 'to': east['id'],
            'baseCost': spacing, 'traffic': traffic_for_edge(eid, PROFILE),
        })
    north = by_coord.get((n['x'], n['y'] + spacing))
    if north:
        eid = edge_id(n['id'], north['id'])
        edges.append({
            'id': eid, 'from': n['id'], 'to': north['id'],
            'baseCost': spacing, 'traffic': traffic_for_edge(eid, PROFILE),
        })

print(f'✅ {len(nodes)} 個の交差点、{len(edges)} 本の道路、{len(delivery_node_ids)} 個の配送先')
print(f'   配送先: {[n["label"] for n in nodes if n["kind"] == "delivery"]}')
print(f'   渋滞分布: {dict((k, sum(1 for e in edges if e["traffic"] == k)) for k in TRAFFIC_MULTIPLIER)}')

## ④ 道路最短距離 (Floyd-Warshall) + 720 順列のコスト表

倉庫と 6 配送先の間の **道路に沿った** 最短距離を計算し、全 720 通りの巡回順序それぞれの総距離を求めます。これが量子回路に渡す「採点ルール」になります。

In [ ]:
node_ids = [n['id'] for n in nodes]
n_nodes = len(node_ids)
idx_of = {nid: i for i, nid in enumerate(node_ids)}
INF = float('inf')
dist = [[INF] * n_nodes for _ in range(n_nodes)]
for i in range(n_nodes):
    dist[i][i] = 0
for e in edges:
    cost = e['baseCost'] * TRAFFIC_MULTIPLIER[e['traffic']]
    i, j = idx_of[e['from']], idx_of[e['to']]
    if cost < dist[i][j]:
        dist[i][j] = cost
        dist[j][i] = cost
for k in range(n_nodes):
    for i in range(n_nodes):
        for j in range(n_nodes):
            alt = dist[i][k] + dist[k][j]
            if alt < dist[i][j]:
                dist[i][j] = alt

depot_idx = idx_of[DEPOT_ID]
delivery_idxs = [idx_of[nid] for nid in delivery_node_ids]

def route_distance(perm):
    total = dist[depot_idx][delivery_idxs[perm[0]]]
    for a, b in zip(perm[:-1], perm[1:]):
        total += dist[delivery_idxs[a]][delivery_idxs[b]]
    total += dist[delivery_idxs[perm[-1]]][depot_idx]
    return total

all_perms = list(permutations(range(len(delivery_idxs))))
costs = np.array([route_distance(p) for p in all_perms])
print(f'✅ {len(all_perms)} 通りの巡回順序、コスト範囲 = [{costs.min():.2f}, {costs.max():.2f}]')
best_idx = int(np.argmin(costs))
print(f'   理論上の最短ルート (古典総当たり): {all_perms[best_idx]} -> {costs[best_idx]:.2f}u')

## ⑤ QAOA 回路を組み立てる

Web アプリの JS エンジン (`src/engine/qaoa.ts`) と同じロジックを Qiskit で書きます。

1. **重ね合わせ**: 10 個の量子ビットに Hadamard をかけ、1024 通りの状態を均等に出現させる
2. **位相付け (γ)**: 各順列のコストを正規化して、短いほど位相が進む対角ユニタリを適用
3. **混ぜる (β)**: 全量子ビットに Rx(2β) を適用
4. **2-3 を reps 回繰り返す**
5. **測定**

(720 順列 < 1024 = 2^10 なので、余った 304 状態には大きな位相を付けて抑制します)

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

num_qubits = int(np.ceil(np.log2(len(all_perms))))  # 10
size = 1 << num_qubits  # 1024

# Cost vector of length 1024: valid = 0..719, invalid = INVALID_PENALTY
INVALID_PENALTY = 1e9
cost_vec = np.full(size, INVALID_PENALTY)
cost_vec[:len(costs)] = costs

# Build phases: short routes get smaller phase angle so they constructively amplify
valid_min, valid_max = costs.min(), costs.max()
span = max(valid_max - valid_min, 1e-9)
phases = np.empty(size)
for i in range(size):
    if cost_vec[i] >= INVALID_PENALTY:
        phases[i] = GAMMA * 1.2  # suppress invalid
    else:
        norm = (cost_vec[i] - valid_min) / span
        phases[i] = GAMMA * (1 - norm)

diag = np.exp(-1j * phases)  # exp(-i*phi) per JS engine convention

qc = QuantumCircuit(num_qubits)
# 1. 重ね合わせ
qc.h(range(num_qubits))
# 2-4. レイヤー
for layer in range(REPS):
    qc.diagonal(diag.tolist(), list(range(num_qubits)))
    for q in range(num_qubits):
        qc.rx(2 * BETA, q)
qc.measure_all()

print(f'✅ {num_qubits} qubits, {REPS} レイヤー')
print(f'   状態空間 = {size} (うち有効な順列 = {len(all_perms)})')
print(qc.draw(output='text', fold=120))

## ⑥ AerSimulator で実行 (4096 ショット)

In [ ]:
simulator = AerSimulator()
tqc = transpile(qc, simulator)
result = simulator.run(tqc, shots=4096).result()
counts = result.get_counts()

# bitstring -> permutation index. Qiskit の bitstring は MSB が一番大きな qubit
def bitstr_to_index(bs):
    return int(bs.replace(' ', ''), 2)

freq = np.zeros(size)
for bs, c in counts.items():
    freq[bitstr_to_index(bs)] = c
freq /= freq.sum()

# 有効順列だけ取り出して降順
valid_freq = freq[:len(all_perms)]
valid_freq = valid_freq / max(valid_freq.sum(), 1e-12)
rank_indices = np.argsort(-valid_freq)

print('🌊 Qiskit 結果 (上位 5)')
for rank, idx in enumerate(rank_indices[:5], 1):
    perm = all_perms[idx]
    labels = [nodes[delivery_idxs[p]]['label'] for p in perm]
    print(f'  #{rank}: 確率 {valid_freq[idx]*100:5.2f}% / 距離 {costs[idx]:6.2f}u')
    print(f'        順序: 倉庫 → {" → ".join(labels)} → 倉庫')

## ⑦ JS エンジンの結果と比較

Web アプリで観測した上位 3 ルートと、Qiskit の上位 3 ルートを並べます。**サンプリング誤差 ±2% 程度**は正常です。

In [ ]:
import pandas as pd

rows = []
max_rows = max(len(JS_TOP_ROUTES), 3)
for rank in range(max_rows):
    qiskit_idx = rank_indices[rank] if rank < len(rank_indices) else None
    qiskit_prob = float(valid_freq[qiskit_idx] * 100) if qiskit_idx is not None else None
    qiskit_dist = float(costs[qiskit_idx]) if qiskit_idx is not None else None
    qiskit_perm = all_perms[qiskit_idx] if qiskit_idx is not None else None

    js_row = JS_TOP_ROUTES[rank] if rank < len(JS_TOP_ROUTES) else None
    js_dist = js_row['distance'] if js_row else None
    js_prob = (js_row['probability'] * 100) if js_row else None

    rows.append({
        'rank': rank + 1,
        'JS 距離': f'{js_dist:.2f}u' if js_dist is not None else '-',
        'JS 確率': f'{js_prob:.2f}%' if js_prob is not None else '-',
        'Qiskit 距離': f'{qiskit_dist:.2f}u' if qiskit_dist is not None else '-',
        'Qiskit 確率': f'{qiskit_prob:.2f}%' if qiskit_prob is not None else '-',
        '一致?': '✅' if (js_dist is not None and qiskit_dist is not None and abs(js_dist - qiskit_dist) < 0.01) else '〜',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

if not JS_TOP_ROUTES:
    print('\nℹ️ Web アプリの結果を貼り付けると比較表が完成します。')

## ⑧ 確率分布の可視化 (Web アプリの「候補バー」と同じビュー)

In [ ]:
import matplotlib.pyplot as plt

top_n = 12
top_probs = valid_freq[rank_indices[:top_n]]
labels_x = [f'#{i+1}' for i in range(top_n)]

fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.bar(labels_x, top_probs * 100, color=['#3DD9C8'] + ['#A5B0C9'] * (top_n - 1))
ax.set_ylabel('確率 (%)')
ax.set_title(f'Qiskit AerSimulator (γ={GAMMA}, β={BETA}, reps={REPS}, {PROFILE})')
ax.axhline(100 / len(all_perms), color='gray', linestyle='--', linewidth=0.8, label=f'一様 (1/{len(all_perms)})')
ax.legend()
plt.tight_layout()
plt.show()

## 🎯 まとめ — チームの成果物

このノートブックの**実行済み URL** (上の `共有 ▸ リンクを取得`) と **⑦の表のスクリーンショット** がチームの成果物です。

プレゼンでは次の 3 点が言えれば OK:

1. 私たちは **どんな街** で **どんな問題** を解いた？ (ストーリーカード参照)
2. つまみをどう調整した？ (γ=___, β=___, reps=___)
3. JS エンジンと Qiskit で結果が **どれくらい一致** した?

---

### 🔭 もっと先へ — IBM Quantum 実機で試す (任意)

本物の量子コンピュータでも動かしたい人は [IBM Quantum](https://quantum.ibm.com/) でアカウント作成 → API Token を取得 → `qiskit-ibm-runtime` を使って実機キューに投入できます。ただし配送先 4 つ程度に縮約しないと実機の qubit 数が足りません。

(ハッカソン中は時間が足りないので、当日の宿題として扱うのがオススメ)